# RNN(记忆之前的状态)
## Ht = tanh(U Xt+W Ht−1+b)
Ht表示隐藏状态，Xt表示输入，U和W是权重矩阵，b是偏置项，tanh是激活函数。
 

In [ ]:
import numpy as np
import torch
import torch.optim as optim
B,E,H = 1,128,3 #批次，词向量维度，隐藏层维度

def prepare_inputs():
    np.random.seed(42)
    vocab = {"播放":0,"周杰伦":1,"的":2,"《稻香》":3}
    tokens = ["播放","周杰伦","的","《稻香》"]
    ids = [vocab[token] for token in tokens]
    V = len(vocab)
    emb_table = np.random.randn(V, E).astype(np.float32) #词向量表，4个128维向量
    x_np = torch.from_numpy(emb_table[ids][None])
    return tokens,x_np

def manual_rnn_numpy(x_np,U_np,W_np):
    B_local,T_local,_ = x_np.shape
    h_prev = torch.zeros((B_local,H),dtype=torch.float32)
    step = []
    for t in range(T_local):
        x_t = x_np[:,t,:]
        h_t = torch.tanh(x_t @ U_np + h_prev @ W_np)
        step.append(h_t)#记录每一步的隐藏状态
        h_prev = h_t
    return torch.stack(step,axis=1),h_prev

tokens,x_np = prepare_inputs()
U_np = torch.randn(E,H, requires_grad=True)
W_np = torch.randn(H,H, requires_grad=True)
y_target = torch.tensor([[0.5, -0.2, 0.8]], dtype=torch.float32)
optimizer = optim.SGD([U_np, W_np], lr=0.01)
learing_rate = 0.01
epochs = 10000
for epoch in range(epochs):
    h_all,h_final = manual_rnn_numpy(x_np,U_np,W_np)
    loss = torch.mean((h_final - y_target)**2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")
        



Epoch 10, Loss: 1.2433
Epoch 20, Loss: 1.2433
Epoch 30, Loss: 1.2433
Epoch 40, Loss: 1.2433
Epoch 50, Loss: 1.2433
Epoch 60, Loss: 1.2433
Epoch 70, Loss: 1.2433
Epoch 80, Loss: 1.2433
Epoch 90, Loss: 1.2433
Epoch 100, Loss: 1.2433
Epoch 110, Loss: 1.2433
Epoch 120, Loss: 1.2433
Epoch 130, Loss: 1.2433
Epoch 140, Loss: 1.2433
Epoch 150, Loss: 1.2433
Epoch 160, Loss: 1.2433
Epoch 170, Loss: 1.2433
Epoch 180, Loss: 1.2433
Epoch 190, Loss: 1.2433
Epoch 200, Loss: 1.2433
Epoch 210, Loss: 1.2433
Epoch 220, Loss: 1.2433
Epoch 230, Loss: 1.2433
Epoch 240, Loss: 1.2433
Epoch 250, Loss: 1.2433
Epoch 260, Loss: 1.2433
Epoch 270, Loss: 1.2433
Epoch 280, Loss: 1.2433
Epoch 290, Loss: 1.2433
Epoch 300, Loss: 1.2433
Epoch 310, Loss: 1.2433
Epoch 320, Loss: 1.2433
Epoch 330, Loss: 1.2433
Epoch 340, Loss: 1.2433
Epoch 350, Loss: 1.2433
Epoch 360, Loss: 1.2433
Epoch 370, Loss: 1.2433
Epoch 380, Loss: 1.2433
Epoch 390, Loss: 1.2433
Epoch 400, Loss: 1.2433
Epoch 410, Loss: 1.2433
Epoch 420, Loss: 1.2433
E

: 

## 使用nn.RNN模块

In [ ]:
import torch.nn as nn
B,E,H = 1,128,3 
def pytorch_rnn_forward(x,U,W):
    rnn = nn.RNN(
        input_size = E,
        hidden_size = H,
        num_layers = 1,#单层RNN
        nonlinearity = 'tanh',#激活函数
        bias=False,
        batch_first = True,#输入的第一个维度是批次
        bidirectional=False#单向RNN
    )
    with torch.no_grad():
        rnn.weight_ih_l0.copy_(U.T)
        rnn.weight_hh_l0.copy_(W.T)#转置
    y, h_n = rnn(x)
    return y, h_n.squeeze(0)#将第一个维度除掉，，使其形状从 (1, batch_size, hidden_size) 变为 (batch_size, hidden_size)

# LSTM(长短期记忆网络)
## 细胞状态和隐藏状态
细胞状态Ct：携带长期记忆信息，允许信息在较长时间内流动。<br>
隐藏状态Ht：携带短期记忆信息，反映当前时间步的输出

## LSTM的门控机制
遗忘门ft:用sigmoid作为激活函数，淘汰旧细胞状态中的不必要信息。
$$
f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)
$$

## 输入门(选择更新的信息)与候选记忆（新的细胞状态）
$$
i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)
$$
$$
\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)
$$

## 更新细胞状态
$$
c_t = (f_t \odot c_{t-1}) + (i_t \odot \tilde{c}_t)
$$

## 输出门(决定输出什么)
$$
o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)
$$
$$
h_t = o_t \odot \tanh(c_t)
$$


In [3]:
import numpy as np
B,E,H = 1,128,3
def prepare_inputs():
    np.random.seed(42)
    vocab = {"播放":0,"周杰伦":1,"的":2,"《稻香》":3}
    tokens = ["播放","周杰伦","的","《稻香》"]
    ids = [vocab[token] for token in tokens]
    V = len(vocab)
    emb_table = np.random.randn(V, E).astype(np.float32)
    x_np = emb_table[ids][None]
    return tokens, x_np
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
def manual_lstm_numpy(x_np, weights):
    U_f, W_f, U_i, W_i, U_c, W_c, U_o, W_o = weights
    B_local,T_local,_ = x_np.shape
    h_prev = np.zeros((B_local,H),dtype=np.float32)
    c_prev = np.zeros((B_local,H),dtype=np.float32)
    step = []
    for t in range(T_local):
        x_t = x_np[:,t,:]
        f_t = sigmoid(x_t @ U_f + h_prev @ W_f)
        t_t = sigmoid(x_t @ U_i + h_prev @ W_i)
        c_i = np.tanh(x_t @ U_c + h_prev @ W_c)
        c_t = f_t * c_prev + t_t * c_i
        O_T = sigmoid(x_t @ U_o + h_prev @ W_o)
        h_t = O_T * np.tanh(c_t)
        step.append(h_t)
        h_prev = h_t
        c_prev = c_t    
    outputs = np.stack(step,axis=1)
    return outputs, h_t, c_t
def main():
    _, x_np = prepare_inputs()
    U_f = np.random.randn(E,H).astype(np.float32)
    W_f = np.random.randn(H,H).astype(np.float32)
    U_i = np.random.randn(E,H).astype(np.float32)
    W_i = np.random.randn(H,H).astype(np.float32)
    U_c = np.random.randn(E,H).astype(np.float32)
    W_c = np.random.randn(H,H).astype(np.float32)
    U_o = np.random.randn(E,H).astype(np.float32)
    W_o = np.random.randn(H,H).astype(np.float32)
    weights = (U_f, W_f, U_i, W_i, U_c, W_c, U_o, W_o)
    print("--- 手写 LSTM (NumPy) ---")
    out_manual_np, hT_manual_np, cT_manual_np = manual_lstm_numpy(x_np, weights)
    print("输入形状:", x_np.shape)
    print("手写输出形状:", out_manual_np.shape)
    print("手写最终隐藏形状:", hT_manual_np.shape)
    print("手写最终细胞形状:", cT_manual_np.shape)


if __name__ == "__main__":
    main()


--- 手写 LSTM (NumPy) ---
输入形状: (1, 4, 128)
手写输出形状: (1, 4, 3)
手写最终隐藏形状: (1, 3)
手写最终细胞形状: (1, 3)


# GRU(门控循环单元)
## 合并细胞状态和隐藏状态
## 重置门
$$
r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)
$$
## 更新门
$$
z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)
$$
## 候选隐藏状态
$$
\tilde{h}_t = \tanh(W_h \cdot [r_t \odot h_{t-1}, x_t] + b_h)
$$
## 最终隐藏状态
$$
h_t = z_t \odot h_{t-1} + (1 - z_t) \odot \tilde{h}_t
$$